In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

from energy_lakehouse.tables import (
    BRONZE_SCHEMA,
    CATALOG,
    DWD_RAW,
    GOLD_SCHEMA,
    POWER_HOURLY,
    SILVER_SCHEMA,
    SMARD_RAW,
    WEATHER_GERMANY_HOURLY,
    WEATHER_STATION_HOURLY,
)

spark.sql(f"USE CATALOG {CATALOG}")
print("catalog is now:", spark.sql("SELECT current_catalog()").collect()[0][0])


def sql_text(value: str) -> str:
    """A SQL string literal. Doubling the quote is how SQL escapes an apostrophe."""
    escaped = value.replace("'", "''")
    return f"'{escaped}'"

In [0]:
SCHEMA_COMMENTS = {
    BRONZE_SCHEMA: (
        "Raw source data as published, plus the run that fetched it. Nothing is cleaned, "
        "renamed or joined here. A bronze row answers one question: what did the source "
        "say, and when did we get it."
    ),
    SILVER_SCHEMA: (
        "One clean, conformed row per hour. UTC hour starts with Berlin labels alongside, "
        "source codes resolved to names, and every table rebuilt in full from bronze on "
        "each run rather than merged incrementally."
    ),
    GOLD_SCHEMA: (
        "Business-level tables built to answer a question rather than to mirror a source. "
        "Deliberately empty until a16."
    ),
}

for schema, comment in SCHEMA_COMMENTS.items():
    spark.sql(f"COMMENT ON SCHEMA {schema} IS {sql_text(comment)}")
    print("commented", schema)

In [0]:
TABLE_COMMENTS = {
    SMARD_RAW: (
        "German electricity generation, consumption and day-ahead price: one row per series "
        "and hour, as published by SMARD. Merged on (filter_id, timestamp_utc), so re-running "
        "a load corrects values instead of duplicating them. "
        "Source: Bundesnetzagentur | SMARD.de, licensed CC BY 4.0."
    ),
    DWD_RAW: (
        "Hourly wind and sunshine observations from eight DWD stations: one row per station, "
        "hour and parameter, read from the recent archives. Merged on "
        "(station_id, timestamp_utc, parameter). "
        "Datenbasis: Deutscher Wetterdienst, Einzelwerte gemittelt. Licensed CC BY 4.0."
    ),
    POWER_HOURLY: (
        "One row per series and hour. The SMARD filter_id is resolved to a series name, group "
        "and unit through a left join, and Berlin calendar labels sit alongside the UTC hour "
        "start. Full rebuild from bronze.smard_raw."
    ),
    WEATHER_STATION_HOURLY: (
        "One row per station and hour on a complete hourly spine, so a missing observation "
        "appears as a null row rather than as no row at all. DWD labels the hour end, so "
        "timestamps are shifted back one hour to match the SMARD hour start. "
        "Full rebuild from bronze.dwd_raw."
    ),
    WEATHER_GERMANY_HOURLY: (
        "One row per hour: an unweighted mean across the eight stations, with the number of "
        "contributing stations and a reliability flag for each measure. Wind direction uses a "
        "circular mean. Full rebuild from silver.weather_station_hourly."
    ),
}

for table, comment in TABLE_COMMENTS.items():
    spark.sql(f"COMMENT ON TABLE {table} IS {sql_text(comment)}")
    print("commented", table)

In [0]:
HOUR_START_UTC = "Start of the hour the row describes, in UTC. The join key between power and weather."
HOUR_START_BERLIN = (
    "The same instant written as Berlin wall-clock time. A label, not an instant: on the "
    "October clock change two different UTC hours carry the same Berlin label."
)
DATE_BERLIN = (
    "Berlin calendar date. Note that (date_berlin, hour_of_day_berlin) is NOT a unique key: "
    "local hour 2 occurs twice on the October clock-change date."
)
HOUR_OF_DAY_BERLIN = "Berlin hour of day, 0 to 23. See date_berlin for why it is not part of a key."
LOAD_ID = "UUID of the notebook run that wrote this row."
FETCHED_AT_RUN = (
    "UTC time the run started. Freshness is measured against max(fetched_at), never against "
    "now(): the two answer different questions."
)

COLUMN_COMMENTS = {
    SMARD_RAW: {
        "filter_id": "SMARD series id, for example 4068 for solar PV. Resolved to a name in silver.",
        "timestamp_utc": "Start of the hour the value applies to, in UTC, exactly as SMARD publishes it.",
        "value": (
            "The published value in the unit of its series. Null where SMARD has padded the "
            "current week ahead of publication, which is why count(*) and count(value) answer "
            "different questions."
        ),
        "source_chunk": (
            "Id of the weekly SMARD file this row came from. Used as the high-water mark so a "
            "load fetches only the weeks bronze does not already hold."
        ),
        "load_id": LOAD_ID,
        "fetched_at": FETCHED_AT_RUN,
    },
    DWD_RAW: {
        "station_id": "Five-digit DWD station id, zero padded, for example 00183 for Arkona.",
        "timestamp_utc": (
            "The hour as DWD labels it, which is the hour END. Silver shifts it back one hour "
            "so it lines up with the SMARD hour start."
        ),
        "parameter": "Which measurement this row holds: wind_speed_ms, wind_direction_deg or sunshine_minutes.",
        "value": "The observation. The DWD missing marker -999 is converted to null when the file is parsed.",
        "quality": "DWD quality level (QN) reported for the measurement.",
        "source_file": "Path of the DWD archive this row was read from.",
        "load_id": LOAD_ID,
        "fetched_at": FETCHED_AT_RUN,
    },
    POWER_HOURLY: {
        "hour_start_utc": HOUR_START_UTC,
        "hour_start_berlin": HOUR_START_BERLIN,
        "date_berlin": DATE_BERLIN,
        "hour_of_day_berlin": HOUR_OF_DAY_BERLIN,
        "filter_id": "SMARD series id, carried unchanged from bronze.",
        "series_name": (
            "Human-readable series name, for example wind_onshore. Null would mean a filter_id "
            "the project does not know; the load check asserts it never is."
        ),
        "series_group": "One of renewable, conventional, price or load.",
        "unit": "MWh for every series except the day-ahead price, which is EUR/MWh.",
        "value": "The published value, carried unchanged from bronze. Null where SMARD has not published yet.",
        "load_id": LOAD_ID,
        "fetched_at": FETCHED_AT_RUN,
    },
    WEATHER_STATION_HOURLY: {
        "hour_start_utc": (
            "Start of the hour, UTC, already shifted back one hour from the DWD hour-end label "
            "so it lines up with power_hourly."
        ),
        "hour_start_berlin": HOUR_START_BERLIN,
        "date_berlin": DATE_BERLIN,
        "hour_of_day_berlin": HOUR_OF_DAY_BERLIN,
        "station_id": "Five-digit DWD station id, zero padded.",
        "station_name": "Station name exactly as the DWD catalogue spells it.",
        "state": "German federal state the station sits in.",
        "wind_speed_ms": "Mean wind speed in metres per second. Null where the station published nothing for this hour.",
        "wind_direction_deg": (
            "Wind direction in degrees, 0 to 360, meaning the direction the wind comes FROM. "
            "Must be averaged circularly, never arithmetically."
        ),
        "sunshine_minutes": (
            "Minutes of sunshine in the hour, 0 to 60. Imputed to 0.0 for the night hours DWD "
            "never published; see sun_is_imputed."
        ),
        "sun_is_imputed": (
            "True where sunshine_minutes was set to 0.0 because the hour is one of the six night "
            "hours (UTC 0, 1, 20, 21, 22, 23) that DWD did not publish before August 2026. Every "
            "other missing value is left null."
        ),
        "wind_quality": "DWD quality level for the wind measurement, carried from bronze.",
        "sun_quality": "DWD quality level for the sunshine measurement, carried from bronze.",
        "fetched_at": (
            "The most recent fetch that touched this station-hour. Null means the hour never "
            "arrived in any file. load_id is deliberately absent here: a station-hour is built "
            "from two archives that can be loaded by different runs, so no single run id "
            "describes the whole row."
        ),
    },
    WEATHER_GERMANY_HOURLY: {
        "hour_start_utc": HOUR_START_UTC,
        "hour_start_berlin": HOUR_START_BERLIN,
        "date_berlin": DATE_BERLIN,
        "hour_of_day_berlin": HOUR_OF_DAY_BERLIN,
        "wind_speed_ms_avg": "Unweighted mean wind speed across the stations that reported this hour.",
        "wind_stations": "How many of the eight stations contributed a wind reading.",
        "wind_is_reliable": (
            "True when at least six stations reported wind. Kept separate from "
            "sunshine_is_reliable because one hour can have eight wind stations and five sun stations."
        ),
        "wind_direction_deg_avg": (
            "Circular mean of the station directions, 0 to 360. The arithmetic mean of 350 and 10 "
            "would be 180, due south, when the wind is in fact from the north."
        ),
        "sunshine_minutes_avg": "Unweighted mean sunshine minutes across the stations that reported this hour.",
        "sunshine_stations": "How many of the eight stations contributed a sunshine value.",
        "sunshine_is_reliable": "True when at least six stations reported sunshine.",
        "sunshine_imputed_stations": (
            "How many of the contributing stations had sunshine imputed to zero for this hour, "
            "so an average built mostly from imputed zeros can be told apart from a measured one."
        ),
        "fetched_at": "The most recent fetch that touched any station-hour inside this hour.",
    },
}

written = 0
for table, columns in COLUMN_COMMENTS.items():
    for column, comment in columns.items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} COMMENT {sql_text(comment)}")
        written += 1
    print(f"{table}: {len(columns)} columns commented")

print("total columns commented:", written)

In [0]:
GRANTS = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `account users`",
    f"GRANT USE SCHEMA ON SCHEMA {GOLD_SCHEMA} TO `account users`",
    f"GRANT SELECT ON SCHEMA {GOLD_SCHEMA} TO `account users`",
]

for statement in GRANTS:
    try:
        spark.sql(statement)
        print("ok  ", statement)
    except Exception as error:
        print("FAIL", statement, "->", error)

print()
print("Nothing is granted on bronze or silver on purpose: a reader gets the published layer only.")
print("SELECT alone would not be enough - a principal also needs USE CATALOG and USE SCHEMA to reach the table.")

In [0]:
%sql
SHOW GRANTS ON SCHEMA gold

In [0]:
%sql
SELECT c.table_schema,
       c.table_name,
       count(*)                                   AS columns,
       count(c.comment)                           AS commented_columns,
       max(CASE WHEN t.comment IS NULL THEN 0 ELSE 1 END) AS table_has_comment
FROM information_schema.columns AS c
JOIN information_schema.tables  AS t
     ON t.table_schema = c.table_schema
    AND t.table_name   = c.table_name
WHERE c.table_schema IN ('bronze', 'silver')
GROUP BY c.table_schema, c.table_name
ORDER BY c.table_schema, c.table_name

In [0]:
DOCS_PATH = "/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/docs/data-dictionary.md"
LAYERS = ("bronze", "silver", "gold")

schemata = {
    row.schema_name: row.comment
    for row in spark.sql(
        "SELECT schema_name, comment FROM information_schema.schemata"
    ).collect()
}
tables = {
    (row.table_schema, row.table_name): row.comment
    for row in spark.sql(
        "SELECT table_schema, table_name, comment FROM information_schema.tables "
        "WHERE table_schema IN ('bronze', 'silver', 'gold')"
    ).collect()
}
columns = spark.sql(
    "SELECT table_schema, table_name, ordinal_position, column_name, full_data_type, "
    "is_nullable, comment FROM information_schema.columns "
    "WHERE table_schema IN ('bronze', 'silver', 'gold') "
    "ORDER BY table_schema, table_name, ordinal_position"
).collect()

lines = [
    "# Data dictionary",
    "",
    f"Every table in the `{CATALOG}` catalog, generated from `information_schema` by",
    "`notebooks/metadata`. Do not edit by hand: re-run that notebook instead.",
    "",
    "It carries no generation date on purpose, so re-running it produces an identical file",
    "unless the schema itself changed, and a diff always means something real.",
    "",
]

for layer in LAYERS:
    lines += [f"## `{CATALOG}.{layer}`", "", (schemata.get(layer) or "_No comment._"), ""]
    in_layer = sorted({name for schema, name in tables if schema == layer})
    if not in_layer:
        lines += ["_No tables yet._", ""]
        continue
    for name in in_layer:
        lines += [f"### `{CATALOG}.{layer}.{name}`", "", (tables[(layer, name)] or "_No comment._"), ""]
        lines += ["| # | Column | Type | Null | Meaning |", "|---|---|---|---|---|"]
        for row in columns:
            if row.table_schema == layer and row.table_name == name:
                nullable = "yes" if row.is_nullable == "YES" else "no"
                meaning = (row.comment or "").replace("|", "\\|")
                lines.append(
                    f"| {row.ordinal_position} | `{row.column_name}` | "
                    f"`{row.full_data_type}` | {nullable} | {meaning} |"
                )
        lines.append("")

markdown = "\n".join(lines)

with open(DOCS_PATH, "w", encoding="utf-8") as handle:
    handle.write(markdown)

print("wrote", DOCS_PATH)
print(len(lines), "lines,", len(columns), "columns documented")